In [1]:
import torch
from torch import nn
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from model.multi_headed_attention import MultiHeadedAttention
from model.vanilla_neural_network import VanillaNeuralNetwork

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
class Decoder_Block(nn.Module):
    def __init__(self, model_dim: int, num_heads: int):
        super().__init__()
        self.mhsa = MultiHeadedAttention(model_dim, num_heads, mask=True)
        self.cmhsa = MultiHeadedAttention(model_dim, num_heads, mask=False)  # Cross-attention
        self.vanilla_nn = VanillaNeuralNetwork(model_dim)
        self.layer_norm_one = nn.LayerNorm(model_dim)
        self.layer_norm_two = nn.LayerNorm(model_dim)
        self.layer_norm_three = nn.LayerNorm(model_dim)

    def forward(self, embedded, encoder_output):
        embedded = embedded + self.mhsa(query=self.layer_norm_one(embedded), key=embedded, value=embedded) # skip connection
        embedded = embedded + self.cmhsa(query=self.layer_norm_two(embedded), key=encoder_output, value=encoder_output) # cross attention skip connection
        embedded = embedded + self.vanilla_nn(self.layer_norm_three(embedded)) # another skip connection
        return embedded

In [8]:
model_dim = 4 # The dimension for embeddings and attention, the same number is often used for both, model_dim > 0
num_heads = 2 # The number of self-attention instances, num_head > 0, and model_dim % num_heads = 0. The input and output shapes do not depend on num_heads
transformer = Decoder_Block(model_dim, num_heads)
transformer = transformer.to(device)
embedded = [
    [[-0.6775, 1.4919, 0.8760, 0.9440],
    [0.4388, 0.5290, -0.2510, -1.2941]],
    [[2.0576, 0.6107, -0.7395, -0.2010],
    [0.4728, 1.0233, -0.9400, 2.0409]]
] # BxTxA
embedded = torch.tensor(embedded, dtype=torch.float32)
embedded = embedded.to(device)

output = transformer(embedded)
print(output)

tensor([[[ 0.1277,  1.7277,  1.2244,  2.5275],
         [ 0.9445,  0.4307, -1.1908, -0.7566]],

        [[ 3.2257,  0.0648, -0.7395, -0.3747],
         [ 1.5563,  0.7258, -1.3456,  2.7039]]], device='cuda:0',
       grad_fn=<AddBackward0>)
